In [ ]:
import pandas as pd
import seaborn as sns
from matplotlib import pyplot as plt

from transformers import pipeline
import numpy as np
from sklearn.metrics import classification_report
import ollama
import json


In [ ]:
df = pd.read_csv("/Users/chandlershortlidge/Desktop/Ironhack/project-dsai-business-case-automated-customer-reviews/data/amazon-customer-reviews/Datafiniti_Amazon_Consumer_Reviews_of_Amazon_Products_May19.csv")

# EDA

In [ ]:
df.info()

In [ ]:
df.shape

In [ ]:
df.columns = df.columns.str.lower()

In [ ]:
df.head()

In [ ]:
df.isnull().sum()

In [ ]:
df = df.drop(columns=["reviews.didpurchase", "reviews.id", "reviews.numhelpful", "reviews.dorecommend"])

In [ ]:
df.isnull().sum()

# Task 1: Review Classification

## Create sentiment categories

In [ ]:
df["reviews.rating"].value_counts()

In [ ]:
sentiment_map = {1: "negative", 2: "negative", 3: "neutral", 4: "positive", 5: "positive"}
df["rating_sentiment"] = df["reviews.rating"].map(sentiment_map)

In [ ]:
df.head()

In [ ]:
df["rating_sentiment"].value_counts()

In [ ]:
percentage_of = round((df["rating_sentiment"].value_counts() / len(df) * 100), 2)

percentage_of

# very imbalanced dataset

## Downsample data for balanced classes

In [ ]:
# Split the dataframe into three: positive, negative, neutral
positive = (df[df["rating_sentiment"] == "positive"])
negative = (df[df["rating_sentiment"] == "negative"])
neutral =  (df[df["rating_sentiment"] == "neutral"])

positive_sampled = positive.sample(n=1206)
negative_sampled = negative.sample(n=1206)

len(positive_sampled)

# concat into one balanced dataframe 

df_balanced = pd.concat([negative_sampled, positive_sampled, neutral])

# confirm
df_balanced["rating_sentiment"].value_counts()

## Choose a classifier model

- 0: Negative
- 1: Neutral
- 2: Positive

In [ ]:
# pick a model 
# CardiffNLP baseline tokenizes the model
classifier = pipeline(task="sentiment-analysis", model="cardiffnlp/twitter-roberta-base-sentiment")

# view a sample 

sample_reviews = df_balanced["reviews.text"].head(10).tolist()

results = classifier(sample_reviews)
results


In [ ]:
sample_sentiment = df_balanced["rating_sentiment"].head(10).tolist
sample_sentiment

In [ ]:
full_results = classifier(df_balanced["reviews.text"].tolist(), truncation=True, max_length=512)

In [ ]:
label_strings = [r["label"] for r in full_results]
# returns a list of strings
label_strings
# turn the list of strings into a Series using pd.Series
labels_for_mapping = pd.Series(label_strings)
# now labels are ready for mapping
labels_for_mapping


In [ ]:
# map the labels into sentiment and create a new column based on the predicted sentiment
label_map = {"LABEL_0": "negative", "LABEL_1": "neutral", "LABEL_2": "positive"}
df_balanced["predicted_sentiment"] = labels_for_mapping.map(label_map).values

In [ ]:
predicted = df_balanced["predicted_sentiment"]
actual = df_balanced["rating_sentiment"]
print(classification_report(actual, predicted))

### Export class-balanced dataset to csv for use in Colab

In [ ]:
df_balanced.to_csv('balanced_reviews.csv', index=False)

# 2. Product Category Clustering

In [ ]:
df["categories"]

In [ ]:
df["categories"].value_counts()

In [ ]:
df["categories"].unique().count()

## Import local LLM (Qwen2.5) with Ollama for clustering

In [ ]:
# view primary categories as benchmark for meta categories
df["primarycategories"].value_counts()

In [ ]:
# Conduct EDA to understand product categories 

office_supplies = df[df["primarycategories"] == "Office Supplies"]
office_supplies

In [ ]:
df["primarycategories"]

In [ ]:
unique_categories = df["categories"].unique()
unique_categories.tolist()

In [ ]:
category_map = {}

# use qwen to loop through the categories and create a dictionary of categories and primary categories 

for cat in unique_categories:
    response = ollama.chat(model="qwen2.5", messages=[
        {"role": "user", "content": f"""Classify this product category: '{cat}' into ONE of: 'Electronics', 'Health & Beauty', 'Tablets & E-readers', 'Home & Kitchen', 'Office Supplies', 'Pet Supplies'. Reply with ONLY the category name. 
         DO NOT invent your own category name. Use ONLY the names provided.
         If the product category contains Health & Beauty or Health and Beauty, then the product category is: Health & Beauty
         Litter boxes are classified as Pet Supplies.
         
         """}
    ])
    category_map[cat] = response["message"]["content"]
    print(response["message"]["content"])

In [ ]:
# turn the dictionary into a dataframe 
# use .items() to split the key:value pairs into a list of tuples so pd.Dataframe can create two colums, key and value

mapping_df = pd.DataFrame(category_map.items(), columns=["categories", "primary_category"])
mapping_df.head()

In [ ]:
df["meta_category"] = df["categories"].map(category_map) # map the LLM's primary categories to the dataframe's categories as "meta_category"

In [ ]:

pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)
df[["categories", "primarycategories", "meta_category"]].value_counts()

In [ ]:
df.columns

## Save the df with meta_categories and rating_sentiment labels as a csv to save tokens & time when reloading

In [ ]:
df.to_csv("amazon_sentiment_categories.csv", index=False)

In [ ]:
new_df = pd.read_csv("/Users/chandlershortlidge/Desktop/Ironhack/project-dsai-business-case-automated-customer-reviews/notebooks/amazon_sentiment_categories.csv")

# Task 3: Review Summarization Using Generative AI

In [ ]:
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)
new_df.groupby(['meta_category', 'name'])['reviews.rating'].mean().sort_values()

In [ ]:
def generate_category_blog(df, category, summaries, model='qwen2.5', n_top=3):
    """
    For a given meta_category:
      - Finds the top N products by review count
      - Finds the worst product by average rating
      - Generates a blog-style summary via Ollama
      - Stores the result in summaries[category_key] and prints it
    """
    # Filter to category
    cat_df = df[df["meta_category"] == category]

    # Top N products by number of reviews
    top_n = (
        cat_df.groupby(['meta_category', 'name'])
        .agg(avg_rating=('reviews.rating', 'mean'), num_reviews=('reviews.rating', 'count'))
        .sort_values(by='num_reviews', ascending=False)
        .head(n_top)
    )

    # Reviews for each top product
    top_names = top_n.index.get_level_values('name').tolist()
    top_reviews = [df[df["name"] == name]["reviews.text"].tolist()[:30] for name in top_names]

    # Worst product by average rating
    worst_product = (
        cat_df.groupby(['meta_category', 'name'])
        .agg(avg_rating=('reviews.rating', 'mean'), num_reviews=('reviews.rating', 'count'))
        .sort_values(by='avg_rating', ascending=True)
        .head(1)
    )
    worst_name = worst_product.index.get_level_values('name')[0]
    worst_reviews = df[df["name"] == worst_name]["reviews.text"].tolist()

    # Build a readable review block for the prompt
    top_reviews_str = "\n\n".join(
        [f"Product {i+1} - {top_names[i]}:\n{top_reviews[i]}" for i in range(len(top_names))]
    )

    # Call Ollama
    response = ollama.chat(model=model, messages=[
        {"role": "user", "content": f"""
Write a short article (like a blog post) about the product category: {category}. The output should include:

- Top {n_top} products {top_n} and key differences between them.
- Reviews and top complaints for each: {top_reviews_str}

-----
You should also include the worst product {worst_product} in the category and why it should be avoided.
Worst product reviews: {worst_reviews}
Do not forget to include this.
----

Make sure the style is like a blog.
"""}
    ])

    # Normalise category name to a dict key (e.g. "Health & Beauty" -> "health_beauty")
    category_key = category.lower().replace(" & ", "_").replace("&", "_").replace(" ", "_")
    summaries[category_key] = response['message']['content']
    print(response['message']['content'])
    return response['message']['content']

In [ ]:
# Generate blog summaries for all categories in one go
categories = [
    "Electronics",
    "Tablets & E-readers",
    "Health & Beauty",
    "Home & Kitchen",
    "Office Supplies",
    "Pet Supplies",
]

summaries = {}
for cat in categories:
    print(f"\n{'='*60}\n{cat}\n{'='*60}")
    generate_category_blog(new_df, cat, summaries)

with open("summaries.json", "w") as f:
    json.dump(summaries, f)